## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:
import numpy as np
import pandas as pd
from math import ceil
from scipy import stats
import statsmodels.stats.api as sms
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

baseline = 0.19
target = 0.20

effect_size = sms.proportion_effectsize(target, baseline)

required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)
required_n = ceil(required_n)

print(f'Розмір ефекту: {effect_size:.4f}')
print(f'Потрібно користувачів в одній групі: {required_n}')
print(f'Потрібно користувачів сумарно (2 групи): {2 * required_n}')

Розмір ефекту: 0.0252
Потрібно користувачів в одній групі: 24638
Потрібно користувачів сумарно (2 групи): 49276


**Відповідь.** «Збільшення утримання на 1 %» розуміємо як зростання з 19 % до 20 %, тобто на 1 відсотковий пункт. Беремо стандартні параметри: рівень значущості 5 % і потужність 80 %.

Потрібно **24 638 користувачів у кожній групі**, тобто **49 276 користувачів сумарно**.

У даних тесту понад 90 тисяч гравців. Отже, вибірки достатньо, щоб помітити такий ефект.

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [2]:
df = pd.read_csv('cookie_cats.csv')
print(df.shape)
display(df.head())

df.groupby('version')['retention_7'].agg(['mean', 'count'])

(90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


,mean,count
version,,
gate_30,0.190201,44700
gate_40,0.182000,45489


**Результат.** Утримання на 7-й день:

| Версія | Ворота | Утримання на 7-й день | Гравців |
|---|---|---|---|
| gate_30 (control) | рівень 30 | 19.02 % | 44 700 |
| gate_40 (treatment) | рівень 40 | 18.20 % | 45 489 |

**Гіпотеза.** Версія **gate_30** (ворота на 30 рівні) дає краще утримання через 7 днів, ніж gate_40. Різниця — близько 0.8 відсоткового пункту. Далі перевіримо, чи вона статистично значуща.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


**Гіпотези z-тесту для двох пропорцій**

- $H_0: p_{control} = p_{treatment}$ — утримання на 7-й день однакове;
- $H_a: p_{control} \ne p_{treatment}$ — утримання різне.

Тест **двосторонній**, бо питаємо, чи *якась* з версій дає кращий результат. Рівень значущості $\alpha = 0.05$.

In [3]:
control_results = df[df['version'] == 'gate_30']['retention_7']
treatment_results = df[df['version'] == 'gate_40']['retention_7']

successes = [control_results.sum(), treatment_results.sum()]
nobs = [control_results.count(), treatment_results.count()]

z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')

z statistic: 3.16
p-value: 0.002
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


**1. Чи є статистично значущою різниця між версіями?**

Так. $p\text{-value} \approx 0.002 < 0.05$, тому відхиляємо $H_0$. Різниця в утриманні на 7-й день статистично значуща. Знак $z \approx 3.16 > 0$ показує, що утримання вище в контрольній групі gate_30.

**2. Чи перетинаються довірчі інтервали?**

Ні. Інтервал control — приблизно [0.1866, 0.1938], інтервал treatment — [0.1785, 0.1855]. Верхня межа treatment (0.1855) нижча за нижню межу control (0.1866).

Це означає, що правдоподібні значення утримання двох версій не збігаються. Неперетин інтервалів — ще одне підтвердження, що різниця справжня, а не випадкова.

**Практичний висновок.** Переносити ворота з 30 на 40 рівень не варто. Це знижує утримання на 7-й день приблизно на 0.8 відсоткового пункту.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


**Гіпотези тесту Хі-квадрат**

- $H_0$: версія гри та утримання на 7-й день **незалежні** — утримання не залежить від того, де стоять ворота;
- $H_a$: між версією гри та утриманням на 7-й день **є залежність**.

Рівень значущості $\alpha = 0.05$.

In [4]:
crosstab = pd.crosstab(df['version'], df['retention_7'])
display(crosstab)

chi2, p, dof, expected = stats.chi2_contingency(crosstab)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected.round(1))
print('Рішення:', 'відхиляємо H0' if p <= 0.05 else 'не відхиляємо H0')

retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[36382.9  8317.1]
 [37025.1  8463.9]]
Рішення: відхиляємо H0


**Висновок.** $\chi^2 \approx 9.96$, $p\text{-value} \approx 0.0016 < 0.05$. **Відхиляємо $H_0$**: утримання на 7-й день залежить від версії гри.

Це той самий результат, що й у z-тесті. Для таблиці 2×2 тест Хі-квадрат рівносильний двосторонньому z-тесту для двох пропорцій: $\chi^2 \approx z^2$ ($3.16^2 \approx 10.0$). Невелика різниця — через поправку Єйтса, яку `chi2_contingency` застосовує до таблиць 2×2.